# Unsupervised Learning: Clustering

This notebook covers:

- K-Means Clustering
- Gaussian Mixture Models (GMM)
- Expectation Maximization (EM)
- DBSCAN
- HDBSCAN
- Hierarchical Clustering

**Dataset:** Iris Dataset

The goal is to understand clustering from both a theoretical and practical perspective using visualizations, evaluation metrics, and algorithm comparisons.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame

print("Head:")
print(df.head())

print("\nInfo:")
print(df.info())

print("\nDescribe:")
print(df.describe())


# Part 1: Theory Recap

Clustering is an unsupervised learning task that groups similar observations together.

### K-Means
- Centroid-based clustering
- Requires number of clusters (K)
- Works best for spherical clusters

### GMM
- Probabilistic clustering
- Produces soft assignments
- Uses Expectation Maximization (EM)

### DBSCAN
- Density-based clustering
- Detects arbitrary shapes
- Handles noise naturally

### HDBSCAN
- Extension of DBSCAN
- Supports varying density clusters

### Hierarchical Clustering
- Builds cluster hierarchy
- Visualized using dendrograms


# Part 2: Data Preprocessing

Most clustering algorithms rely on distance calculations.
Therefore, feature scaling is important to prevent one feature from dominating others.


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.iloc[:, :4].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)


# Part 3: K-Means From Scratch

This implementation demonstrates the core idea:
1. Initialize centroids.
2. Assign points.
3. Update centroids.
4. Repeat until convergence.


In [ ]:
def kmeans_scratch(X, k=3, max_iter=100):
    centroids = X[:k].copy()

    for _ in range(max_iter):
        distances = np.linalg.norm(X[:, None] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)

        new_centroids = np.array([
            X[labels == i].mean(axis=0)
            for i in range(k)
        ])

        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return labels, centroids

scratch_labels, scratch_centroids = kmeans_scratch(X_scaled)

print("Clusters:", np.unique(scratch_labels))


# Part 4: Library Implementations

Scikit-learn implementations are optimized and widely used in production systems.


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

dbscan = DBSCAN(eps=0.8, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X_scaled)

print("Models trained successfully")


# Part 5: HDBSCAN

HDBSCAN is an advanced density-based clustering algorithm.

If the package is unavailable, install:

```python
pip install hdbscan
```


In [ ]:
try:
    import hdbscan

    hdb = hdbscan.HDBSCAN(min_cluster_size=5)
    hdb_labels = hdb.fit_predict(X_scaled)

    print("HDBSCAN clusters:", np.unique(hdb_labels))
except Exception as e:
    print("HDBSCAN not installed:", e)


# Part 6: Evaluation Metrics

Silhouette Score measures:
- Cluster compactness
- Cluster separation

Higher is generally better.


In [ ]:
from sklearn.metrics import silhouette_score

results = {}

results["KMeans"] = silhouette_score(X_scaled, kmeans_labels)
results["GMM"] = silhouette_score(X_scaled, gmm_labels)
results["Hierarchical"] = silhouette_score(X_scaled, agg_labels)

if len(set(dbscan_labels)) > 1:
    results["DBSCAN"] = silhouette_score(X_scaled, dbscan_labels)

for k,v in results.items():
    print(k, round(v,4))


# Part 7: Cluster Visualizations

Visual comparison often reveals strengths and weaknesses that metrics alone may miss.


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=kmeans_labels)
plt.title("K-Means Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=gmm_labels)
plt.title("GMM Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=agg_labels)
plt.title("Hierarchical Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=dbscan_labels)
plt.title("DBSCAN Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


# Part 8: Dendrogram Visualization

A dendrogram shows how clusters are merged during hierarchical clustering.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(10,5))
dendrogram(linked)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Samples")
plt.ylabel("Distance")
plt.show()


# Part 9: Hyperparameter Experiment

We evaluate different values of K using Silhouette Score.


In [ ]:
scores = []
k_values = [2,3,4,5,6,7]

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_scaled)
    scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(6,4))
plt.plot(k_values, scores, marker='o', label='Silhouette Score')
plt.title("Choosing Optimal K")
plt.xlabel("Number of Clusters")
plt.ylabel("Score")
plt.legend()
plt.show()


# Part 10: Algorithm Comparison

| Algorithm | Strength | Limitation |
|------------|-----------|------------|
| K-Means | Fast | Needs K |
| GMM | Soft assignments | More computation |
| DBSCAN | Handles noise | Sensitive parameters |
| HDBSCAN | Variable densities | Extra library |
| Hierarchical | Dendrogram insight | Expensive on large data |


# Part 11: Interview Corner

### Why does K-Means fail on non-spherical clusters?
Because it uses distance to centroids and assumes roughly spherical cluster boundaries.

### Difference between GMM and K-Means?
K-Means provides hard assignments while GMM provides probabilities.

### Why use DBSCAN?
It detects arbitrary-shaped clusters and identifies outliers naturally.

### What is EM?
An iterative optimization method used to estimate GMM parameters.

### What is a dendrogram?
A tree-like structure showing cluster merges in hierarchical clustering.


# Part 12: Key Takeaways

1. Clustering discovers hidden structure in unlabeled data.
2. K-Means is simple and efficient.
3. GMM provides probabilistic clustering.
4. DBSCAN and HDBSCAN are useful for density-based clusters.
5. Hierarchical clustering offers interpretable cluster relationships.
6. Evaluation metrics and visualizations should be used together.
